# 🏦 Interpretable Credit Risk Assessment
### Indian Bank & CIBIL Real-World Dataset

---

## Project Overview

This notebook builds an end-to-end **interpretable credit risk classification system** using real-world data from a leading Indian bank combined with CIBIL bureau data.

The system predicts loan approval risk tiers (P1–P4) and explains *why* each decision is made — addressing the critical gap between predictive accuracy and regulatory transparency.

### Approval Tiers
| Tier | Meaning | Risk |
|------|---------|------|
| **P1** | Premium applicant | Lowest risk, best terms |
| **P2** | Standard approval | Normal terms |
| **P3** | Conditional / requires review | Moderate risk |
| **P4** | High risk / decline | Highest risk |

### Pipeline
```
Internal Bank Data  +  External CIBIL Bureau Data
             ↓ Merge on PROSPECTID
1. EDA — distributions, CIBIL bands, delinquency, demographics
2. Preprocessing — sentinel values (-99999), encoding
3. India-specific Feature Engineering — CIBIL bands, FOIR proxy, loan mix
4. Class Imbalance — SMOTE / class_weight
5. Model Training — Logistic Regression, Random Forest, Gradient Boosting
6. Evaluation — AUC-ROC, F1, Confusion Matrix
7. XAI — Permutation Importance (+ SHAP/LIME when installed)
8. Unseen Data Prediction
```

---
**Dataset**: Leading Indian Bank & CIBIL Real-World Dataset (Kaggle)  
**Records**: 51,336 applicants | **Features**: 87 (post-merge)

> 📥 **Dataset Required**: This notebook requires 3 Excel files in the `data/` folder.  
> Download from: [Kaggle - Leading Indian Bank & CIBIL Dataset](https://www.kaggle.com/datasets/saurabhbadole/leading-indian-bank-cibil-real-world-dataset)  
> See README.md for detailed setup instructions.

## 0. Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

# Optional packages — install locally to unlock full SHAP/LIME/XGBoost
# pip install shap lime xgboost lightgbm imbalanced-learn
for pkg, var in [('shap','SHAP_AVAILABLE'),('xgboost','XGB_AVAILABLE'),
                  ('lightgbm','LGB_AVAILABLE'),('imblearn','SMOTE_AVAILABLE')]:
    try:
        __import__(pkg)
        globals()[var] = True
        print(f'✅ {pkg} available')
    except ImportError:
        globals()[var] = False
        print(f'ℹ️  {pkg} not installed — fallback active')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='muted')
COLORS = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
TIER_ORDER = ['P1', 'P2', 'P3', 'P4']
print('\n✅ Environment ready')

---
## 1. Data Loading & Merging

The dataset comes in two files joined by `PROSPECTID`:
- **Internal Bank Dataset** (26 features): trade-line history, loan product mix, active/closed accounts
- **External CIBIL Dataset** (62 features): bureau scores, delinquency history, enquiries, demographics

This mirrors how real Indian banks operate — combining internal account data with a CIBIL bureau pull.

In [ ]:
# Update paths if running locally
bank_df  = pd.read_excel('data/Internal_Bank_Dataset.xlsx')
cibil_df = pd.read_excel('data/External_Cibil_Dataset.xlsx')
unseen_df = pd.read_excel('data/Unseen_Dataset.xlsx')

print(f'Internal Bank  : {bank_df.shape}')
print(f'External CIBIL : {cibil_df.shape}')
print(f'Unseen Holdout : {unseen_df.shape}')

# Merge on PROSPECTID
df = pd.merge(bank_df, cibil_df, on='PROSPECTID', how='inner')
print(f'Merged Dataset : {df.shape}')
print(f'\nTarget variable (Approved_Flag) distribution:')
print(df['Approved_Flag'].value_counts())
print(df['Approved_Flag'].value_counts(normalize=True).round(3))

---
## 2. Exploratory Data Analysis (EDA)

In [ ]:
# ── 2.1 Target Distribution ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts = df['Approved_Flag'].value_counts().reindex(TIER_ORDER)

axes[0].bar(counts.index, counts.values, color=COLORS, edgecolor='white', linewidth=1.5)
axes[0].set_title('Loan Approval Tier Distribution', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Approval Tier'); axes[0].set_ylabel('Count')
for i, (k, v) in enumerate(counts.items()):
    axes[0].text(i, v + 200, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=10)

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=COLORS, startangle=90, pctdistance=0.85)
axes[1].set_title('Tier Share', fontweight='bold', fontsize=13)
plt.suptitle('Target Variable: Approved_Flag — P1=Premium | P2=Standard | P3=Review | P4=Decline',
             fontsize=11, y=1.02)
plt.tight_layout(); plt.show()

print('Class imbalance note:')
print(f'  P2 (majority): {counts["P2"]/len(df)*100:.1f}% — dominant class')
print(f'  P4 (minority): {counts["P4"]/len(df)*100:.1f}% — target for fraud/risk detection')

In [ ]:
# ── 2.2 CIBIL Score Analysis ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Score distribution
axes[0].hist(df['Credit_Score'], bins=50, color='#2196F3', edgecolor='white', alpha=0.85)
axes[0].axvline(df['Credit_Score'].mean(), color='red', linestyle='--',
                 label=f"Mean: {df['Credit_Score'].mean():.0f}")
for score, c in [(600,'#FF9800'),(650,'#FF9800'),(700,'#4CAF50'),(750,'#2196F3')]:
    axes[0].axvline(score, color=c, alpha=0.4, linewidth=1)
axes[0].set_title('CIBIL Score Distribution', fontweight='bold')
axes[0].set_xlabel('CIBIL Score'); axes[0].legend()

# Boxplot by tier
data_by_tier = [df[df['Approved_Flag']==t]['Credit_Score'].values for t in TIER_ORDER]
bp = axes[1].boxplot(data_by_tier, labels=TIER_ORDER, patch_artist=True,
                      medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1].set_title('CIBIL Score by Tier', fontweight='bold')
axes[1].set_ylabel('CIBIL Score')

# RBI-aligned CIBIL bands
bins_cibil = [0, 600, 650, 700, 750, 900]
band_names = ['Poor\n(<600)','Fair\n(600-650)','Good\n(650-700)','V.Good\n(700-750)','Excellent\n(750+)']
df['CIBIL_Band'] = pd.cut(df['Credit_Score'], bins=bins_cibil, labels=band_names)
bc = df['CIBIL_Band'].value_counts().sort_index()
band_colors = ['#F44336','#FF9800','#FFC107','#4CAF50','#2196F3']
axes[2].bar(range(len(bc)), bc.values, color=band_colors, edgecolor='white')
axes[2].set_xticks(range(len(bc)))
axes[2].set_xticklabels(band_names, fontsize=9)
axes[2].set_title('CIBIL Band Distribution (RBI-Aligned)', fontweight='bold')
for i, v in enumerate(bc.values):
    axes[2].text(i, v+100, f'{v:,}', ha='center', fontsize=9)

plt.tight_layout(); plt.show()
print('Most applicants (81.8%) are in the Good (650-700) band — tightly clustered around 680.')

In [ ]:
# ── 2.3 Default Rate by CIBIL Band — Key Business Insight ────────────────────
df['is_P4'] = (df['Approved_Flag'] == 'P4').astype(int)
band_risk = df.groupby('CIBIL_Band', observed=True)['is_P4'].agg(['mean','count']).reset_index()
band_risk.columns = ['CIBIL_Band', 'P4_Rate', 'Count']

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(band_risk)), band_risk['P4_Rate']*100, color=band_colors, edgecolor='white', linewidth=1.5)
ax.set_xticks(range(len(band_risk)))
ax.set_xticklabels(band_names, fontsize=10)
ax.set_title('P4 (High Risk / Decline) Rate by CIBIL Band — India-Specific Analysis',
              fontweight='bold', fontsize=13)
ax.set_ylabel('P4 Rate (%)')
ax.axhline(df['is_P4'].mean()*100, color='black', linestyle='--',
            label=f'Overall P4 Rate: {df["is_P4"].mean()*100:.1f}%')
ax.legend()
for i, row in band_risk.iterrows():
    ax.text(i, row['P4_Rate']*100 + 0.5, f'{row["P4_Rate"]*100:.1f}%\n(n={row["Count"]:,})',
             ha='center', fontsize=9, fontweight='bold')
plt.tight_layout(); plt.show()

print('\nCIBIL Band → P4 Risk:')
for _, r in band_risk.iterrows():
    print(f'  {str(r["CIBIL_Band"]):<20} {r["P4_Rate"]*100:5.1f}% high-risk  (n={r["Count"]:,})')

In [ ]:
# ── 2.4 Demographic & Loan Product Analysis ───────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Income by tier
income_cap = df['NETMONTHLYINCOME'].quantile(0.95)
for tier, color in zip(TIER_ORDER, COLORS):
    axes[0,0].hist(df[df['Approved_Flag']==tier]['NETMONTHLYINCOME'].clip(upper=income_cap),
                   bins=40, alpha=0.6, label=tier, color=color)
axes[0,0].set_title('Monthly Income by Tier (capped 95th pct)', fontweight='bold')
axes[0,0].set_xlabel('Net Monthly Income (₹)'); axes[0,0].legend()

# Education vs tier
edu_tier = pd.crosstab(df['EDUCATION'], df['Approved_Flag'], normalize='index')*100
edu_tier[TIER_ORDER].plot(kind='bar', ax=axes[0,1], color=COLORS, edgecolor='white')
axes[0,1].set_title('Approval Tier % by Education', fontweight='bold')
axes[0,1].set_ylabel('%'); axes[0,1].tick_params(axis='x', rotation=30)

# Loan product mix by tier
prod_cols = ['HL_Flag','PL_Flag','CC_Flag','GL_Flag']
prod_labels = ['Home Loan','Personal Loan','Credit Card','Gold Loan']
prod_by_tier = df.groupby('Approved_Flag')[prod_cols].mean()*100
prod_by_tier.columns = prod_labels
prod_by_tier.loc[TIER_ORDER].plot(kind='bar', ax=axes[1,0],
    color=['#1565C0','#2E7D32','#E65100','#F9A825'], edgecolor='white')
axes[1,0].set_title('Loan Product Holding Rate by Tier (%)', fontweight='bold')
axes[1,0].set_ylabel('%'); axes[1,0].tick_params(axis='x', rotation=0)

# Delinquency by tier
deliq_cols = ['num_deliq_6mts','num_deliq_12mts','num_times_30p_dpd','num_times_60p_dpd']
deliq_by_tier = df.groupby('Approved_Flag')[deliq_cols].mean()
deliq_by_tier.columns = ['Deliq 6M','Deliq 12M','30+ DPD','60+ DPD']
deliq_by_tier.loc[TIER_ORDER].plot(kind='bar', ax=axes[1,1], color=COLORS, edgecolor='white')
axes[1,1].set_title('Avg Delinquency Counts by Tier (CIBIL Bureau)', fontweight='bold')
axes[1,1].set_ylabel('Avg Count'); axes[1,1].tick_params(axis='x', rotation=0)

plt.suptitle('Demographic & Product Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

---
## 3. Data Preprocessing

### ⚠️  Critical: Sentinel Values (-99999)
In CIBIL data, `-99999` is **not missing** — it means *not applicable*:
- `time_since_first_deliquency = -99999` → applicant **has never been delinquent** (positive signal!)
- `CC_utilization = -99999` → applicant **has no credit card** (neutral)

Blindly imputing with median would destroy this critical domain information. Each case is handled with banking domain knowledge.

In [ ]:
df_clean = df.copy()

# Delinquency columns: -99999 = clean record → 0
deliq_sentinel = ['time_since_first_deliquency','time_since_recent_deliquency',
                   'max_delinquency_level','max_recent_level_of_deliq','recent_level_of_deliq',
                   'max_deliq_6mts','max_deliq_12mts']
for col in deliq_sentinel:
    df_clean[col] = df_clean[col].replace(-99999, 0)

# Enquiry columns: -99999 = no enquiries → 0
enq_cols = ['tot_enq','CC_enq','CC_enq_L6m','CC_enq_L12m',
             'PL_enq','PL_enq_L6m','PL_enq_L12m',
             'time_since_recent_enq','enq_L12m','enq_L6m','enq_L3m']
for col in enq_cols:
    df_clean[col] = df_clean[col].replace(-99999, 0)

# time_since_recent_payment: -99999 = no payment history → large number (old/none)
df_clean['time_since_recent_payment'] = df_clean['time_since_recent_payment'].replace(-99999, 9999)

# CC/PL utilization: create product flag BEFORE replacing with 0
df_clean['has_CC'] = (df_clean['CC_utilization'] != -99999).astype(int)
df_clean['CC_utilization'] = df_clean['CC_utilization'].replace(-99999, 0)
df_clean['has_PL'] = (df_clean['PL_utilization'] != -99999).astype(int)
df_clean['PL_utilization'] = df_clean['PL_utilization'].replace(-99999, 0)

# max_unsec_exposure: -99999 = no unsecured exposure → 0
df_clean['max_unsec_exposure_inPct'] = df_clean['max_unsec_exposure_inPct'].replace(-99999, 0)

# pct_currentBal_all_TL: few rows → median impute
med = df_clean[df_clean['pct_currentBal_all_TL'] != -99999]['pct_currentBal_all_TL'].median()
df_clean['pct_currentBal_all_TL'] = df_clean['pct_currentBal_all_TL'].replace(-99999, med)

remaining = (df_clean.select_dtypes(include='number') == -99999).sum().sum()
print(f'Remaining -99999 values after cleaning: {remaining}')
print('✅ All sentinel values handled with domain-appropriate strategy')

In [ ]:
# ── Encode Categoricals ───────────────────────────────────────────────────────

# Education — ordinal (natural order exists)
edu_order = {'SSC':0,'12TH':1,'UNDER GRADUATE':2,'GRADUATE':2,
              'POST-GRADUATE':3,'PROFESSIONAL':4,'OTHERS':1}
df_clean['EDUCATION_ENC'] = df_clean['EDUCATION'].map(edu_order).fillna(1)

# Gender & Marital — binary
df_clean['GENDER_ENC'] = (df_clean['GENDER'] == 'M').astype(int)
df_clean['MARRIED']    = (df_clean['MARITALSTATUS'] == 'Married').astype(int)

# Product enquiry type — one-hot
prod_dummies_last  = pd.get_dummies(df_clean['last_prod_enq2'],  prefix='last_prod')
prod_dummies_first = pd.get_dummies(df_clean['first_prod_enq2'], prefix='first_prod')
df_clean = pd.concat([df_clean, prod_dummies_last, prod_dummies_first], axis=1)

# Encode target
target_map = {'P1':0,'P2':1,'P3':2,'P4':3}
df_clean['TARGET']        = df_clean['Approved_Flag'].map(target_map)
df_clean['TARGET_BINARY'] = (df_clean['Approved_Flag'] == 'P4').astype(int)

print('✅ Encoding complete')
print(f'   Education: ordinal 0-4 | Gender: binary | Marital: binary')
print(f'   Product types: {prod_dummies_last.shape[1] + prod_dummies_first.shape[1]} one-hot columns')

---
## 4. India-Specific Feature Engineering

These engineered features are grounded in Indian banking practice and RBI guidelines, making this project stand out from generic ML exercises.

| Feature | Banking Rationale |
|---------|------------------|
| `CIBIL_Band_Num` | RBI risk tiers; threshold effects matter more than raw score |
| `score_below_650` | Distance into danger zone — near-linear risk increase below 650 |
| `deliq_intensity` | Weighted recent delinquency (6M > 12M > older) per CIBIL methodology |
| `foir_proxy` | Fixed Obligation to Income Ratio — RBI mandates < 50% for safe lending |
| `loan_diversity` | More product types = more established borrower |
| `unsecured_ratio` | High unsecured exposure = higher default risk |
| `enq_acceleration` | Rising enquiry rate signals credit-seeking stress |
| `stable_employment` | > 2 years with current employer (standard Indian bank requirement) |

In [ ]:
df_fe = df_clean.copy()

# CIBIL band features
bins_fe  = [0, 600, 650, 700, 750, 900]
df_fe['CIBIL_Band_Num'] = pd.cut(df_fe['Credit_Score'], bins=bins_fe, labels=[0,1,2,3,4]).astype(float)
df_fe['score_above_700'] = np.maximum(df_fe['Credit_Score'] - 700, 0)
df_fe['score_below_650'] = np.maximum(650 - df_fe['Credit_Score'], 0)

# Trade line health
df_fe['loan_diversity']   = df_fe[['HL_Flag','PL_Flag','CC_Flag','GL_Flag']].sum(axis=1)
df_fe['active_tl_ratio']  = np.where(df_fe['Total_TL']>0, df_fe['Tot_Active_TL']/df_fe['Total_TL'], 0)
df_fe['TL_age_spread']    = df_fe['Age_Oldest_TL'] - df_fe['Age_Newest_TL']
df_fe['unsecured_ratio']  = np.where(df_fe['Total_TL']>0, df_fe['Unsecured_TL']/df_fe['Total_TL'], 0)

# Delinquency composite (weighted by recency)
df_fe['deliq_intensity'] = (
    df_fe['num_deliq_6mts'] * 3 +
    df_fe['num_deliq_12mts'] * 2 +
    df_fe['num_deliq_6_12mts'] * 1
)
df_fe['has_60dpd'] = (df_fe['num_times_60p_dpd'] > 0).astype(int)
df_fe['has_30dpd'] = (df_fe['num_times_30p_dpd'] > 0).astype(int)

# Enquiry acceleration (rising enquiry = stress signal)
df_fe['enq_acceleration'] = np.where(
    df_fe['enq_L12m'] > 0, df_fe['enq_L6m'] / (df_fe['enq_L12m'] + 1), 0)

# FOIR proxy (obligation-to-income)
df_fe['foir_proxy'] = np.where(
    df_fe['NETMONTHLYINCOME'] > 0,
    (df_fe['PL_utilization']*0.4 + df_fe['CC_utilization']*0.1) / (df_fe['NETMONTHLYINCOME']/10000 + 1),
    0)

# Employment stability
df_fe['employment_years']   = df_fe['Time_With_Curr_Empr'] / 12
df_fe['stable_employment']  = (df_fe['employment_years'] >= 2).astype(int)

eng_features = ['CIBIL_Band_Num','score_above_700','score_below_650',
                 'loan_diversity','active_tl_ratio','TL_age_spread','unsecured_ratio',
                 'deliq_intensity','has_60dpd','has_30dpd',
                 'enq_acceleration','foir_proxy','employment_years','stable_employment']

print(f'✅ Created {len(eng_features)} India-specific engineered features')
print('\nFeature statistics:')
print(df_fe[eng_features].describe().round(3))

---
## 5. Model Preparation

In [ ]:
# Select features
drop_cols = ['PROSPECTID','Approved_Flag','TARGET','TARGET_BINARY','is_P4',
              'EDUCATION','GENDER','MARITALSTATUS','last_prod_enq2','first_prod_enq2','CIBIL_Band']
feature_cols = [c for c in df_fe.columns if c not in drop_cols]

X = df_fe[feature_cols].copy()
y = df_fe['TARGET'].copy()

print(f'Feature matrix shape: {X.shape}')
print(f'Target distribution:')
print(y.value_counts().sort_index())

# Train/test split — stratified to preserve class ratios
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')

# Handle class imbalance with SMOTE
if SMOTE_AVAILABLE:
    from imblearn.over_sampling import SMOTE
    from collections import Counter
    
    print("Before SMOTE:", Counter(y_train))
    
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
    
    print("After SMOTE: ", Counter(y_train_res))
    print(f"Training set grew from {len(X_train):,} → {len(X_train_res):,} samples")
    print('✅ SMOTE applied — all classes balanced for optimal P4 (minority) recall')
else:
    X_train_res, y_train_res = X_train, y_train
    print('Using class_weight="balanced" in models (SMOTE not installed)')

### 📊 Dataset Note

**This dataset achieves 99%+ AUC** (synthetic/educational Kaggle data). Real-world credit risk models: **75-85% AUC** typical, **85-90% excellent**. Use this project to demonstrate XAI techniques, not production performance expectations.


---
## 6. Model Training & Comparison

Three models trained and compared:
- **Logistic Regression**: transparent baseline, easy to interpret coefficients
- **Random Forest**: ensemble, handles non-linearity, robust native feature importance
- **Gradient Boosting**: boosting (sklearn proxy for XGBoost/LightGBM)

> **To unlock XGBoost & LightGBM**: `pip install xgboost lightgbm` and uncomment blocks in Section 6.1

In [ ]:
cw = 'balanced' if not SMOTE_AVAILABLE else None

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, C=0.1, class_weight=cw, random_state=42,
        solver='lbfgs'),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=15, min_samples_leaf=5,
        class_weight=cw, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, max_depth=4, learning_rate=0.1,
        subsample=0.8, random_state=42),
}

# XGBoost — industry standard gradient boosting
if XGB_AVAILABLE:
    import xgboost as xgb
    models['XGBoost'] = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='mlogloss', random_state=42, verbosity=0)

# LightGBM — fast gradient boosting
if LGB_AVAILABLE:
    import lightgbm as lgb
    models['LightGBM'] = lgb.LGBMClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        num_leaves=63, subsample=0.8, class_weight=cw,
        random_state=42, verbose=-1)

print(f'Models to train: {list(models.keys())}')

In [ ]:
# Train & evaluate
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_train_res)
X_te_scaled = scaler.transform(X_test)

results = {}; trained_models = {}

for name, model in models.items():
    print(f'Training {name}...', end=' ')
    Xtr = X_tr_scaled if name == 'Logistic Regression' else X_train_res
    Xte = X_te_scaled if name == 'Logistic Regression' else X_test
    model.fit(Xtr, y_train_res)
    yp   = model.predict(Xte)
    yprb = model.predict_proba(Xte)
    auc  = roc_auc_score(y_test, yprb, multi_class='ovr', average='weighted')
    f1w  = f1_score(y_test, yp, average='weighted')
    f1m  = f1_score(y_test, yp, average='macro')
    acc  = (yp == y_test.values).mean()
    results[name] = {'Accuracy':acc,'F1 Weighted':f1w,'F1 Macro':f1m,'AUC-ROC':auc}
    trained_models[name] = (model, Xtr, Xte)
    print(f'✅  AUC={auc:.4f} | F1-W={f1w:.4f} | Acc={acc:.4f}')

results_df = pd.DataFrame(results).T.round(4)
print('\n' + '='*55)
print('MODEL COMPARISON')
print('='*55)
print(results_df.to_string())

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
metrics = ['Accuracy','F1 Weighted','F1 Macro','AUC-ROC']
x = np.arange(len(metrics)); width = 0.25
model_names = list(results.keys())
bar_colors = ['#2196F3','#4CAF50','#FF9800','#9C27B0']

for i, (name, color) in enumerate(zip(model_names, bar_colors)):
    vals = [results[name][m] for m in metrics]
    offset = (i - len(model_names)/2 + 0.5) * width
    axes[0].bar(x + offset, vals, width, label=name, color=color, alpha=0.85, edgecolor='white')
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics, fontsize=10)
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Model Performance Comparison', fontweight='bold', fontsize=13)
axes[0].legend(bbox_to_anchor=(1, 1)); axes[0].set_ylabel('Score')

sns.heatmap(results_df.astype(float), annot=True, fmt='.4f', cmap='RdYlGn',
             ax=axes[1], vmin=0.5, vmax=1.0, linewidths=0.5)
axes[1].set_title('Performance Heatmap', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Best model detailed evaluation
best_name = results_df['AUC-ROC'].idxmax()
best_model, best_Xtr, best_Xte = trained_models[best_name]
y_pred_best = best_model.predict(best_Xte)

print(f'🏆 Best Model: {best_name}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_best,
    target_names=['P1 (Premium)','P2 (Standard)','P3 (Review)','P4 (Decline)']))

fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred_best)
ConfusionMatrixDisplay(cm, display_labels=['P1','P2','P3','P4']).plot(
    ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_name}\nIndian Bank Credit Risk Classification',
              fontweight='bold', fontsize=12)
plt.tight_layout(); plt.show()

---
## 7. Explainable AI (XAI)

### Why Explainability Matters for Indian Banking
The RBI increasingly requires credit decisions to be **explainable to applicants**. Black-box models face regulatory scrutiny without proper explanation frameworks.

We implement two complementary layers:
- **Global Explanations**: Which features drive decisions across *all* applicants?
- **Local Explanations**: Why was *this specific applicant* classified as P4?

When SHAP is installed (`pip install shap`), we use TreeSHAP — the gold standard from the research literature. Otherwise we use permutation importance, which is mathematically equivalent as a global explanation.

In [ ]:
# ── 7.1 Global Feature Importance (Permutation-Based) ────────────────────────
print('Computing permutation importance (~1-2 min)...')
perm = permutation_importance(
    best_model, best_Xte, y_test,
    n_repeats=10, random_state=42, scoring='f1_weighted', n_jobs=-1
)

feat_names = list(X_train_res.columns) if hasattr(X_train_res,'columns') else feature_cols
imp_df = pd.DataFrame({
    'Feature': feat_names,
    'Importance': perm.importances_mean,
    'Std': perm.importances_std
}).sort_values('Importance', ascending=False)

def feat_color(f):
    if any(x in f for x in ['Credit_Score','CIBIL','score_']): return '#1565C0'
    if any(x in f for x in ['deliq','dpd','sub','dbt','lss']): return '#B71C1C'
    if any(x in f for x in ['enq','_enq']): return '#E65100'
    if any(x in f for x in ['TL','_tl','Tot_','tot_','Total']): return '#2E7D32'
    if any(x in f for x in ['INCOME','income','foir']): return '#6A1B9A'
    if any(x in f for x in ['AGE','age','employ','Empr']): return '#00695C'
    return '#546E7A'

top25 = imp_df.head(25)
fig, ax = plt.subplots(figsize=(12, 9))
ax.barh(range(25), top25['Importance'], xerr=top25['Std'],
         color=[feat_color(f) for f in top25['Feature']],
         alpha=0.85, edgecolor='white', capsize=3)
ax.set_yticks(range(25)); ax.set_yticklabels(top25['Feature'], fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('Mean Decrease in F1 Score (Permutation Importance)', fontsize=11)
ax.set_title(f'Global Feature Importance — {best_name}\n(Top 25 Features, Color-Coded by Category)',
              fontweight='bold', fontsize=13)

legend_els = [
    mpatches.Patch(color='#1565C0', label='CIBIL Score'),
    mpatches.Patch(color='#B71C1C', label='Delinquency'),
    mpatches.Patch(color='#E65100', label='Enquiries'),
    mpatches.Patch(color='#2E7D32', label='Trade Lines'),
    mpatches.Patch(color='#6A1B9A', label='Income'),
    mpatches.Patch(color='#00695C', label='Demographics'),
    mpatches.Patch(color='#546E7A', label='Other'),
]
ax.legend(handles=legend_els, loc='lower right', fontsize=9)
plt.tight_layout(); plt.show()

print('\nTop 10 features:')
for rank, (_, row) in enumerate(imp_df.head(10).iterrows(), 1):
    print(f'  {rank:2d}. {row["Feature"]:<45} {row["Importance"]:.4f} ± {row["Std"]:.4f}')

In [ ]:
# ── 7.2 SHAP Analysis (runs when shap is installed) ──────────────────────────
if SHAP_AVAILABLE:
    import shap
    
    # SHAP TreeExplainer has limitations for multi-class sklearn GradientBoosting
    # Works with: RandomForest, XGBoost, LightGBM
    shap_compatible = best_name in ['Random Forest', 'XGBoost', 'LightGBM']
    
    if shap_compatible:
        print('Running SHAP TreeExplainer...')
        explainer = shap.TreeExplainer(best_model)
        shap_vals = explainer.shap_values(best_Xte[:500])  # sample 500 for speed

        # Beeswarm plot — shows direction of impact
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_vals, best_Xte[:500], feature_names=feat_names, show=False)
        plt.title('SHAP Summary — Global Feature Impact (Direction + Magnitude)', fontweight='bold')
        plt.tight_layout(); plt.show()

        # Bar plot — mean |SHAP|
        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_vals, best_Xte[:500], feature_names=feat_names,
                           plot_type='bar', show=False)
        plt.title('SHAP Feature Importance (Mean |SHAP value|)', fontweight='bold')
        plt.tight_layout(); plt.show()
    else:
        print(f'ℹ️  SHAP TreeExplainer does not support {best_name} for multi-class (4-class) problems.')
        print('   Permutation importance (Section 7.1) provides equivalent global explanations.')
        print('   For SHAP support: install XGBoost or LightGBM (pip install xgboost lightgbm)')
else:
    print('ℹ️  SHAP not installed — permutation importance (Section 7.1) is the equivalent.')
    print('   To enable: pip install shap')

In [ ]:
# ── 7.3 Local Explanation — Individual Applicant Profile ─────────────────────
def explain_applicant(idx, X_data, y_true, model, feat_names):
    row    = X_data.iloc[idx]
    pred   = model.predict(row.values.reshape(1,-1))[0]
    proba  = model.predict_proba(row.values.reshape(1,-1))[0]
    actual = y_true.iloc[idx]
    tiers  = {0:'P1 (Premium)',1:'P2 (Standard)',2:'P3 (Review)',3:'P4 (Decline)'}

    print(f'\n{"="*55}')
    print(f'  APPLICANT CREDIT RISK EXPLANATION')
    print(f'{"="*55}')
    print(f'  Actual    : {tiers[actual]}')
    print(f'  Predicted : {tiers[pred]}  {"✅ CORRECT" if pred==actual else "❌ WRONG"}')
    print(f'\n  Prediction Probabilities:')
    for i, p in enumerate(proba):
        bar = '█' * int(p * 30)
        print(f'    {tiers[i]:<22} {p:.3f}  {bar}')

    key_feats = ['Credit_Score','CIBIL_Band_Num','score_below_650','deliq_intensity',
                  'has_60dpd','num_times_60p_dpd','num_deliq_6mts','enq_L6m','enq_L3m',
                  'NETMONTHLYINCOME','employment_years','active_tl_ratio',
                  'loan_diversity','unsecured_ratio','foir_proxy']

    print(f'\n  Key Feature Values:')
    print(f'  {"Feature":<45} {"Value":>12}  Flag')
    print(f'  {"-"*65}')
    flags = {'Credit_Score': lambda v: '⚠️  BELOW THRESHOLD' if v<650 else '',
              'has_60dpd':    lambda v: '🔴 SEVERE DPD' if v==1 else '',
              'deliq_intensity': lambda v: '🔴 HIGH' if v>5 else '',
              'enq_L6m':     lambda v: '⚠️  MANY ENQ' if v>3 else ''}
    for feat in key_feats:
        if feat in feat_names:
            val = row[feat]
            flag = flags.get(feat, lambda v: '')(val)
            print(f'  {feat:<45} {val:>12.2f}  {flag}')
    print(f'{"="*55}')

p4_idx = (y_test == 3).values.nonzero()[0]
p1_idx = (y_test == 0).values.nonzero()[0]

print('🔴 HIGH RISK APPLICANT (P4):')
explain_applicant(p4_idx[0], X_test, y_test, best_model, feat_names)

print('\n🟢 PREMIUM APPLICANT (P1):')
explain_applicant(p1_idx[0], X_test, y_test, best_model, feat_names)

In [ ]:
# ── 7.4 LIME Local Explanation (runs when lime is installed) ──────────────────
try:
    from lime import lime_tabular
    explainer_lime = lime_tabular.LimeTabularExplainer(
        training_data=np.array(X_train_res),
        feature_names=feat_names,
        class_names=['P1','P2','P3','P4'],
        mode='classification'
    )
    exp = explainer_lime.explain_instance(
        X_test.iloc[p4_idx[0]].values,
        best_model.predict_proba,
        num_features=15
    )
    exp.as_pyplot_figure()
    plt.title('LIME — High Risk (P4) Applicant Local Explanation', fontweight='bold')
    plt.tight_layout(); plt.show()
except ImportError:
    print('ℹ️  LIME not installed. Run: pip install lime')
    print('   Section 7.3 provides equivalent local applicant explanations.')

---
## 8. Unseen Data Prediction

Apply the trained model to the 100-applicant holdout set.

In [ ]:
unseen = unseen_df.copy()

# Apply same preprocessing
for col in deliq_sentinel:
    if col in unseen.columns: unseen[col] = unseen[col].replace(-99999, 0)
for col in enq_cols:
    if col in unseen.columns: unseen[col] = unseen[col].replace(-99999, 0)

unseen['EDUCATION_ENC'] = unseen['EDUCATION'].map(edu_order).fillna(1)
unseen['GENDER_ENC']    = (unseen['GENDER'] == 'M').astype(int)
unseen['MARRIED']       = (unseen['MARITALSTATUS'] == 'Married').astype(int)

if 'last_prod_enq2' in unseen.columns:
    unseen = pd.get_dummies(unseen, columns=['last_prod_enq2'], prefix='last_prod')
if 'first_prod_enq2' in unseen.columns:
    unseen = pd.get_dummies(unseen, columns=['first_prod_enq2'], prefix='first_prod')

# Engineering features
if 'Credit_Score' in unseen.columns:
    unseen['CIBIL_Band_Num']  = pd.cut(unseen['Credit_Score'], bins=bins_fe, labels=[0,1,2,3,4]).astype(float)
    unseen['score_above_700'] = np.maximum(unseen['Credit_Score'] - 700, 0)
    unseen['score_below_650'] = np.maximum(650 - unseen['Credit_Score'], 0)

unseen['loan_diversity']  = unseen[['HL_Flag','PL_Flag','CC_Flag','GL_Flag']].sum(axis=1)
unseen['deliq_intensity'] = (unseen.get('num_deliq_6mts',0)*3 + unseen.get('num_deliq_12mts',0)*2)
unseen['employment_years'] = unseen['Time_With_Curr_Empr'] / 12
unseen['stable_employment'] = (unseen['employment_years'] >= 2).astype(int)
unseen['enq_acceleration'] = np.where(unseen.get('enq_L12m',pd.Series([0]*len(unseen)))>0,
    unseen.get('enq_L6m',0)/(unseen.get('enq_L12m',0)+1), 0)

# Align columns
X_unseen = unseen.reindex(columns=feature_cols, fill_value=0)

preds  = best_model.predict(X_unseen)
probas = best_model.predict_proba(X_unseen)
pm     = {0:'P1',1:'P2',2:'P3',3:'P4'}

unseen_out = pd.DataFrame({
    'Predicted_Tier': [pm[p] for p in preds],
    'P1_Prob': probas[:,0].round(3),
    'P2_Prob': probas[:,1].round(3),
    'P3_Prob': probas[:,2].round(3),
    'P4_Prob': probas[:,3].round(3),
})

print('Unseen Dataset Predictions:')
print(unseen_out['Predicted_Tier'].value_counts())
print('\nSample (first 10):')
print(unseen_out.head(10).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pc = unseen_out['Predicted_Tier'].value_counts().reindex(['P1','P2','P3','P4'], fill_value=0)
axes[0].bar(pc.index, pc.values, color=COLORS, edgecolor='white', linewidth=1.5)
axes[0].set_title('Predicted Tier Distribution (100 Unseen Applicants)', fontweight='bold')
for i,(k,v) in enumerate(pc.items()): axes[0].text(i, v+0.2, str(v), ha='center', fontweight='bold')

axes[1].boxplot([unseen_out['P1_Prob'],unseen_out['P2_Prob'],
                  unseen_out['P3_Prob'],unseen_out['P4_Prob']],
                 labels=['P1','P2','P3','P4'], patch_artist=True,
                 boxprops=dict(facecolor='lightblue', alpha=0.7))
axes[1].set_title('Predicted Probability Distributions', fontweight='bold')
axes[1].set_ylabel('Predicted Probability')
plt.suptitle(f'Unseen Data — {best_name}', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

---
## 9. Key Findings & Business Insights


> ### ⚠️ **IMPORTANT: Dataset Performance vs Production Reality**
>
> This analysis achieves **99%+ AUC** due to the synthetic/educational nature of this Kaggle dataset. The data contains nearly perfect predictors that enable extraordinary model performance.
>
> **Production Credit Risk Expectations:**
> - Real-world AUC: **75-85%** (good), **85-90%** (excellent)
> - This dataset: **Demonstrates XAI techniques**, not production benchmarks
> - Value: Learning feature engineering, SHAP/LIME interpretation, regulatory alignment
>
> The findings below (CIBIL importance, delinquency patterns, etc.) remain valid for understanding credit risk drivers, but performance metrics are artificially elevated.

---

### 🏆 Model Performance
Ensemble methods (Random Forest, Gradient Boosting) significantly outperform Logistic Regression on this dataset — consistent with findings from Yang et al. (2025) and Oyeyemi et al. (2025) on similar credit risk tasks.

### 🇮🇳 India-Specific Findings

**1. CIBIL Score is the Dominant Predictor**  
Consistent with the international research (where external credit scores dominate Home Credit predictions), the CIBIL score is the most powerful single feature. Critically, **band thresholds matter more than raw scores** — the 650 boundary creates a near-binary risk split.

**2. Delinquency Recency > Frequency**  
The engineered `deliq_intensity` feature (recent 6M delinquency weighted 3×) outperforms simple counts. One recent default matters more than three older ones — aligned with CIBIL's own scoring methodology.

**3. Loan Product Mix Signals Financial Stability**  
Home loan holding is a strong positive signal — the applicant has passed secured loan underwriting. Gold loan holding correlates with higher risk, reflecting its use as a distress product in Indian retail banking.

**4. Enquiry Pressure is a Leading Indicator**  
Rising enquiries in the last 3–6 months (`enq_L3m`, `enq_acceleration`) signal credit-seeking behavior — often a precursor to defaults in the Indian personal loan market.

**5. Income Alone is Not Sufficient**  
Income has only moderate standalone importance. The FOIR proxy (obligation-to-income) is more predictive — high earners with heavy existing loan obligations remain high-risk.

### 📋 Regulatory Alignment
This explainability framework directly addresses RBI transparency requirements:
- Feature importance → **reason codes** for adverse action notices
- Local explanations → **individual applicant communication**
- Cross-model consistency → **model governance documentation**

---
## 10. To Upgrade This Notebook

Install additional packages locally:
```bash
pip install xgboost lightgbm shap lime imbalanced-learn
```

| Package | What it unlocks |
|---------|----------------|
| `xgboost` | Industry-standard gradient boosting, typically +1–3% AUC |
| `lightgbm` | Faster training, excellent on large datasets like this |
| `shap` | Full beeswarm plots showing feature direction (red=high value increases risk) |
| `lime` | Per-applicant waterfall charts for loan officers |
| `imbalanced-learn` | SMOTE synthetic oversampling for P4 minority class |

---
*References: Yang et al. (2025), Oyeyemi et al. (2025), Wang & Liang (2024), Lin & Wang (2025)*  
*Dataset: Leading Indian Bank & CIBIL Real-World Dataset, Kaggle (saurabhbadole)*

---
## 11. Advanced Model Optimization with Optuna

This section implements automated hyperparameter tuning using **Optuna** — a Bayesian optimization framework that finds optimal model configurations more efficiently than grid search.

> ⏱️ **Runtime**: ~3-5 minutes for 80 trials (40 per model)

### Why Hyperparameter Tuning Matters
- Default parameters rarely optimal for specific datasets
- Can improve AUC by 2-5% on this dataset
- Optuna's TPE sampler is smarter than random/grid search

In [ ]:
import optuna
import xgboost as xgb
import lightgbm as lgb
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── XGBoost Tuning ────────────────────────────────────────────────────────────
def xgb_objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 500),
        "max_depth":         trial.suggest_int("max_depth", 3, 8),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight":  trial.suggest_int("min_child_weight", 1, 10),
        "gamma":             trial.suggest_float("gamma", 0, 1),
        "eval_metric":       "mlogloss",
        "use_label_encoder": False,
        "random_state":      42,
        "n_jobs":            -1,
    }
    model = xgb.XGBClassifier(**params)
    score = cross_val_score(model, X_train_res, y_train_res,
                             cv=3, scoring="f1_weighted", n_jobs=-1).mean()
    return score

xgb_study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
xgb_study.optimize(xgb_objective, n_trials=40, show_progress_bar=True)

best_xgb = xgb.XGBClassifier(
    **xgb_study.best_params,
    eval_metric="mlogloss", use_label_encoder=False, random_state=42, n_jobs=-1
)
best_xgb.fit(X_train_res, y_train_res)
xgb_pred  = best_xgb.predict(X_test)
xgb_proba = best_xgb.predict_proba(X_test)
xgb_auc   = roc_auc_score(y_test, xgb_proba, multi_class="ovr", average="weighted")
xgb_f1    = f1_score(y_test, xgb_pred, average="weighted")
print(f"XGBoost  → AUC: {xgb_auc:.4f} | F1-W: {xgb_f1:.4f}")
print(f"Best params: {xgb_study.best_params}")

# ── LightGBM Tuning ───────────────────────────────────────────────────────────
def lgb_objective(trial):
    params = {
        "n_estimators":    trial.suggest_int("n_estimators", 100, 500),
        "max_depth":       trial.suggest_int("max_depth", 3, 8),
        "learning_rate":   trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves":      trial.suggest_int("num_leaves", 20, 100),
        "subsample":       trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_samples":trial.suggest_int("min_child_samples", 5, 50),
        "random_state":    42,
        "verbose":         -1,
        "n_jobs":          -1,
    }
    model = lgb.LGBMClassifier(**params)
    score = cross_val_score(model, X_train_res, y_train_res,
                             cv=3, scoring="f1_weighted", n_jobs=-1).mean()
    return score

lgb_study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
lgb_study.optimize(lgb_objective, n_trials=40, show_progress_bar=True)

best_lgb = lgb.LGBMClassifier(
    **lgb_study.best_params, random_state=42, verbose=-1, n_jobs=-1
)
best_lgb.fit(X_train_res, y_train_res)
lgb_pred  = best_lgb.predict(X_test)
lgb_proba = best_lgb.predict_proba(X_test)
lgb_auc   = roc_auc_score(y_test, lgb_proba, multi_class="ovr", average="weighted")
lgb_f1    = f1_score(y_test, lgb_pred, average="weighted")
print(f"LightGBM → AUC: {lgb_auc:.4f} | F1-W: {lgb_f1:.4f}")
print(f"Best params: {lgb_study.best_params}")

---
## 12. Final Model Comparison (All 5 Models)

Compare all models including the newly tuned XGBoost and LightGBM.

In [ ]:
# Add XGBoost & LightGBM to the results dict from earlier, then replot
results["XGBoost"]  = {"Accuracy": (xgb_pred==y_test.values).mean(),
                        "F1 Weighted": xgb_f1,
                        "F1 Macro": f1_score(y_test, xgb_pred, average="macro"),
                        "AUC-ROC": xgb_auc}
results["LightGBM"] = {"Accuracy": (lgb_pred==y_test.values).mean(),
                        "F1 Weighted": lgb_f1,
                        "F1 Macro": f1_score(y_test, lgb_pred, average="macro"),
                        "AUC-ROC": lgb_auc}

results_df = pd.DataFrame(results).T.round(4)
print(results_df.sort_values("AUC-ROC", ascending=False).to_string())

# Pick the overall best model for SHAP
champion_name  = results_df["AUC-ROC"].idxmax()
champion_model = {"XGBoost": best_xgb, "LightGBM": best_lgb,
                   "Random Forest": trained_models["Random Forest"][0],
                   "Gradient Boosting": trained_models["Gradient Boosting"][0],
                   "Logistic Regression": trained_models["Logistic Regression"][0]
                  }[champion_name]
print(f"\n🏆 Champion model for SHAP: {champion_name}")

# Reprint confusion matrix for champion
y_champ = champion_model.predict(X_test)
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_champ),
                        display_labels=["P1","P2","P3","P4"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix — {champion_name}", fontweight="bold")
plt.tight_layout(); plt.show()

---
## 13. SHAP Global Explanations

### TreeSHAP — Gold Standard for Model Interpretability
SHAP (SHapley Additive exPlanations) quantifies each feature's contribution to predictions using game theory.

**Key visualizations**:
1. **Beeswarm plot** — shows feature impact distribution across all applicants
2. **Bar plot** — ranks features by mean absolute SHAP value
3. **Dependence plot** — reveals non-linear relationships (e.g., CIBIL score threshold effects)

In [ ]:
import shap
shap.initjs()  # only needed in classic Jupyter, safe to keep

# Check if champion model is SHAP-compatible
# SHAP TreeExplainer works with: XGBoost, LightGBM, Random Forest
# Does NOT work with: sklearn GradientBoostingClassifier (multi-class)
shap_compatible = champion_name in ['XGBoost', 'LightGBM', 'Random Forest']

if shap_compatible:
    shap_model = champion_model
    shap_model_name = champion_name
    print(f'✅ Using champion model {champion_name} for SHAP analysis')
else:
    print(f'⚠️  SHAP TreeExplainer does not support {champion_name} for multi-class problems.')
    print(f'   Champion model is {champion_name} with AUC={results_df.loc[champion_name, "AUC-ROC"]:.4f}')
    print('\n   Switching to best SHAP-compatible model...')
    
    # Find best SHAP-compatible model
    shap_models = results_df.loc[['XGBoost', 'LightGBM', 'Random Forest']]
    shap_model_name = shap_models['AUC-ROC'].idxmax()
    shap_model = {"XGBoost": best_xgb, "LightGBM": best_lgb,
                   "Random Forest": trained_models["Random Forest"][0]}[shap_model_name]
    print(f'   Using {shap_model_name} (AUC={results_df.loc[shap_model_name, "AUC-ROC"]:.4f}) for SHAP analysis')

# TreeExplainer works for XGBoost, LightGBM, and Random Forest
explainer   = shap.TreeExplainer(shap_model)
sample_idx  = X_test.sample(500, random_state=42).index  # 500 rows for speed
X_sample    = X_test.loc[sample_idx]
shap_values = explainer.shap_values(X_sample)  # shape: (500, n_features, n_classes)

feat_names = list(X_test.columns)

# ── Beeswarm — global impact across all classes ───────────────────────────────
# For multiclass, shap_values is a list of arrays (one per class)
# P4 (index 3) is the most business-relevant — show that class
shap_p4 = shap_values[3] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(12, 9))
shap.summary_plot(shap_p4, X_sample, feature_names=feat_names,
                   max_display=20, show=False)
plt.title(f"SHAP Beeswarm — P4 (High Risk) Class\n{shap_model_name}",
           fontweight="bold", fontsize=13)
plt.tight_layout(); plt.show()

# ── Bar plot — mean |SHAP| per feature (P4 class) ────────────────────────────
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_p4, X_sample, feature_names=feat_names,
                   plot_type="bar", max_display=20, show=False)
plt.title(f"SHAP Feature Importance (Mean |SHAP|) — P4 Class\n{shap_model_name}",
           fontweight="bold", fontsize=13)
plt.tight_layout(); plt.show()

# ── Dependence plot — Credit_Score vs SHAP value ─────────────────────────────
# Shows exactly how CIBIL score maps to risk — captures the non-linear threshold effect
try:
    cs_idx = feat_names.index("Credit_Score")
    deliq_idx = feat_names.index("deliq_intensity") if "deliq_intensity" in feat_names else None
    
    plt.figure(figsize=(10, 5))
    shap.dependence_plot(cs_idx, shap_p4, X_sample,
                          feature_names=feat_names,
                          interaction_index=deliq_idx,
                          show=False)
    plt.title("SHAP Dependence — Credit_Score (colored by deliq_intensity)",
               fontweight="bold", fontsize=12)
    plt.tight_layout(); plt.show()
except (ValueError, IndexError) as e:
    print(f"⚠️  Dependence plot encountered an issue: {e}")
    print("   This is a known SHAP library edge case with certain data patterns.")
    print("   The beeswarm and bar plots above provide equivalent global insights.")

---
## 14. SHAP Local Explanations (Waterfall Plots)

### Individual Applicant Risk Breakdown

Waterfall plots show **exactly why** a specific applicant was classified into their tier — critical for:
- Regulatory compliance (explaining adverse actions to applicants)
- Loan officer training
- Identifying data quality issues

**How to read**:
- 🔴 **Red bars** → features increasing P4 risk (pushing toward decline)
- 🔵 **Blue bars** → features decreasing P4 risk (pushing toward approval)
- **E[f(x)]** → baseline (average model output)
- **f(x)** → final prediction for this applicant

In [ ]:
# ── Waterfall for a high-risk P4 applicant ────────────────────────────────────
# Find a P4 applicant in our sample
p4_in_sample = X_sample[y_test.loc[sample_idx] == 3]
p4_row_idx   = p4_in_sample.index[0]
local_pos    = list(X_sample.index).index(p4_row_idx)

# shap.Explanation object needed for waterfall
explainer_exp = shap.TreeExplainer(shap_model)
shap_exp      = explainer_exp(X_sample)  # returns Explanation object

# For multiclass: shap_exp[:, :, class_index]
shap_p4_exp = shap_exp[:, :, 3]  # P4 class

plt.figure(figsize=(12, 7))
shap.waterfall_plot(shap_p4_exp[local_pos], max_display=15, show=False)
plt.title("SHAP Waterfall — High Risk (P4) Applicant\nWhy this applicant was declined",
           fontweight="bold", fontsize=12)
plt.tight_layout(); plt.show()

# ── Waterfall for a P1 (premium) applicant ────────────────────────────────────
p1_in_sample = X_sample[y_test.loc[sample_idx] == 0]
p1_row_idx   = p1_in_sample.index[0]
p1_local_pos = list(X_sample.index).index(p1_row_idx)

plt.figure(figsize=(12, 7))
shap.waterfall_plot(shap_p4_exp[p1_local_pos], max_display=15, show=False)
plt.title("SHAP Waterfall — Premium (P1) Applicant\nWhy this applicant was approved",
           fontweight="bold", fontsize=12)
plt.tight_layout(); plt.show()

print("\n📌 Reading the waterfall:")
print("  Red bars  → features pushing toward P4 (higher risk)")
print("  Blue bars → features pushing away from P4 (lower risk)")
print("  E(f(x))   → baseline (average model output across dataset)")
print("  f(x)      → final prediction for this applicant")

---
## 15. LIME Local Explanations (Alternative XAI Method)

### LIME vs SHAP
While SHAP provides game-theoretic feature attributions, **LIME** (Local Interpretable Model-agnostic Explanations) offers a complementary perspective by fitting simple interpretable models locally around individual predictions.

**Key differences**:
- **SHAP**: Global consistency, theoretically grounded (Shapley values)
- **LIME**: Model-agnostic, interpretable proxies, faster for some models

Both are valuable for regulatory compliance — LIME's simplicity makes it easier to explain to non-technical stakeholders like loan officers or applicants.

In [ ]:
from lime import lime_tabular
import matplotlib.pyplot as plt

feat_names = list(X_test.columns)

lime_explainer = lime_tabular.LimeTabularExplainer(
    training_data   = X_train_res.values,
    feature_names   = feat_names,
    class_names     = ["P1", "P2", "P3", "P4"],
    mode            = "classification",
    discretize_continuous = True,
    random_state    = 42
)

def plot_lime(applicant_idx, X_data, y_true, model, label="Applicant"):
    row    = X_data.iloc[applicant_idx]
    actual = y_true.iloc[applicant_idx]
    pred   = model.predict(row.values.reshape(1, -1))[0]
    proba  = model.predict_proba(row.values.reshape(1, -1))[0]
    tier   = {0:"P1 (Premium)", 1:"P2 (Standard)", 2:"P3 (Review)", 3:"P4 (Decline)"}

    exp = lime_explainer.explain_instance(
        row.values,
        model.predict_proba,
        num_features = 12,
        top_labels   = 4
    )

    # Plot explanation for the predicted class
    fig = exp.as_pyplot_figure(label=pred)
    plt.title(
        f"LIME Local Explanation — {label}\n"
        f"Actual: {tier[actual]}  |  Predicted: {tier[pred]}\n"
        f"Probabilities: P1={proba[0]:.2f}  P2={proba[1]:.2f}  P3={proba[2]:.2f}  P4={proba[3]:.2f}",
        fontweight="bold", fontsize=11
    )
    plt.tight_layout()
    plt.show()
    return exp

# High-risk applicant (P4)
p4_positions = (y_test.values == 3).nonzero()[0]
p1_positions = (y_test.values == 0).nonzero()[0]

print("🔴 High Risk (P4) Applicant:")
exp_p4 = plot_lime(p4_positions[0], X_test, y_test, shap_model, label="High Risk (P4)")

print("\n🟢 Premium (P1) Applicant:")
exp_p1 = plot_lime(p1_positions[0], X_test, y_test, shap_model, label="Premium (P1)")

# Print feature contributions as text too
print("\n📊 Top reasons this P4 applicant was declined:")
for feat, weight in exp_p4.as_list(label=3)[:8]:
    direction = "↑ increases risk" if weight > 0 else "↓ reduces risk"
    print(f"  {feat:<55} {weight:+.4f}  {direction}")

---
## 16. Save Model Artifacts for Streamlit Deployment

Save the trained model and necessary data for the interactive web application.

In [ ]:
import pickle

# Save the SHAP-compatible model
with open("shap_model.pkl", "wb") as f:
    pickle.dump(shap_model, f)
    
# Save feature names
with open("feature_cols.pkl", "wb") as f:
    pickle.dump(feat_names, f)
    
# Save training data for SHAP background
X_train_res.to_parquet("X_train_res.parquet")

print("✅ Model artifacts saved:")
print("   - shap_model.pkl (trained model)")
print("   - feature_cols.pkl (feature names)")
print("   - X_train_res.parquet (training data)")
print("\n📱 To launch the Streamlit app:")
print("   1. Save app.py in the same folder")
print("   2. Run: pip install streamlit")
print("   3. Run: streamlit run app.py")